# Fairness_Evaluation.ipynb — Group 15 | 7PAM2033
# Purpose : Evaluate demographic bias and fairness across all three models.
#           Compute fairness metrics before and after bias mitigation.
#           This directly addresses the project KPIs:
#             - FPR difference between groups ≤ 5%
#             - FNR difference between groups ≤ 5%
#             - Measurable demographic parity improvement after mitigation
#             - Measurable equalized odds improvement after mitigation

In [ ]:
# ================================================================================
# CELL 1 — Imports
# ================================================================================

import os
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})

logger.info("Fairness evaluation started.")

In [ ]:
# ================================================================================
# CELL 2 — Paths and settings
# ================================================================================

RESULTS_DIR  = Path("../results")
PLOTS_DIR    = Path("../results/plots")
CLINICAL_CSV = Path("../Data/Clinical_Data.csv")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Sensitive demographic columns used for fairness analysis
SENSITIVE_COLS = ["Age", "Gender", "Ethnicity", "EducationLevel"]

# Human-readable labels for encoded demographic values
GENDER_MAP    = {0: "Female",       1: "Male"}
ETHNICITY_MAP = {0: "Caucasian",    1: "Afr. American",
                 2: "Asian",        3: "Other"}
EDUCATION_MAP = {0: "None",         1: "High School",
                 2: "Bachelor's",   3: "Higher"}

# Fairness KPI thresholds from project spec
FPR_THRESHOLD = 0.05   # max allowed FPR difference between groups
FNR_THRESHOLD = 0.05   # max allowed FNR difference between groups

In [ ]:
# ================================================================================
# CELL 3 — Load predictions from all three models
# ================================================================================
# Each model notebook saved its predictions to CSV — we load them here
# so we can evaluate fairness without rerunning the models.

def load_predictions(path, model_name):
    """Load model predictions and check the file exists."""
    if not Path(path).exists():
        logger.warning("%s predictions not found: %s", model_name, path)
        return None
    df = pd.read_csv(path)
    logger.info("%s loaded: %d rows", model_name, len(df))
    return df

m1_preds = load_predictions(RESULTS_DIR / "model1_predictions.csv", "Model 1")
m2_preds = load_predictions(RESULTS_DIR / "model2_predictions.csv", "Model 2")
m3_preds = load_predictions(RESULTS_DIR / "model3_predictions.csv", "Model 3")

# Check which models are available
available = {k: v for k, v in {
    "Model 1 CNN": m1_preds,
    "Model 2 EfficientNet": m2_preds,
    "Model 3 Hybrid": m3_preds
}.items() if v is not None}

print(f"Models available for fairness evaluation: {list(available.keys())}")


In [ ]:
# ================================================================================
# CELL 4 — Load clinical data for demographic information
# ================================================================================
# The clinical dataset has the demographic columns (Age, Gender, Ethnicity,
# EducationLevel) that we use as sensitive attributes for fairness analysis.
# For models 1 and 2 (MRI only) we attach demographic data from the clinical
# dataset by proportion matching — since patients differ between datasets.

df_clinical = pd.read_csv(CLINICAL_CSV)
drop_cols   = [c for c in ["PatientID", "DoctorInCharge"] if c in df_clinical.columns]
df_clinical.drop(columns=drop_cols, inplace=True)

print(f"Clinical data: {df_clinical.shape}")
print(f"Sensitive columns available: "
      f"{[c for c in SENSITIVE_COLS if c in df_clinical.columns]}")


In [ ]:
# ================================================================================
# CELL 5 — Helper functions for fairness metrics
# ================================================================================
# These functions compute the core fairness metrics used in the project.
# All metrics are computed per demographic group and compared across groups.

def compute_fairness_metrics(y_true, y_pred, group_labels, group_name):
    """
    Compute FPR, FNR, precision and recall per demographic group.
    Returns a DataFrame with one row per group.

    FPR (False Positive Rate) = FP / (FP + TN)
    FNR (False Negative Rate) = FN / (FN + TP)
    High FPR in a group → model over-diagnoses that group
    High FNR in a group → model misses diagnoses in that group
    """
    # Convert to binary for FPR/FNR (diagnosed vs not diagnosed)
    # For multi-class: diagnosed = any demented class (not NonDemented)
    if y_true.max() > 1:
        y_true_bin = (y_true > 0).astype(int)   # 0=healthy, 1=any dementia
        y_pred_bin = (y_pred > 0).astype(int)
    else:
        y_true_bin = y_true
        y_pred_bin = y_pred

    rows = []
    for group in sorted(group_labels.unique()):
        mask = group_labels == group
        if mask.sum() < 10:   # skip groups with too few samples
            continue

        yt = y_true_bin[mask]
        yp = y_pred_bin[mask]

        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

        fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr  = fn / (fn + tp) if (fn + tp) > 0 else 0
        acc  = accuracy_score(yt, yp)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0

        rows.append({
            "Group":     group,
            "N":         int(mask.sum()),
            "FPR":       round(fpr, 4),
            "FNR":       round(fnr, 4),
            "Accuracy":  round(acc, 4),
            "Precision": round(prec, 4),
            "Recall":    round(rec, 4),
        })

    df = pd.DataFrame(rows).set_index("Group")
    df[f"FPR_diff from mean"] = (df["FPR"] - df["FPR"].mean()).round(4)
    df[f"FNR_diff from mean"] = (df["FNR"] - df["FNR"].mean()).round(4)
    return df


def check_kpi(metric_df, threshold, metric_name):
    """Check if FPR or FNR differences between groups meet the KPI target."""
    diff = metric_df[metric_name].max() - metric_df[metric_name].min()
    met  = diff <= threshold
    print(f"  {metric_name} max gap: {diff:.4f}  "
          f"(KPI: ≤ {threshold})  {'✅ MET' if met else '❌ NOT MET'}")
    return met


def demographic_parity(y_pred_bin, group_labels):
    """
    Demographic parity — the positive prediction rate should be similar
    across all demographic groups. Large differences indicate bias.
    """
    rates = {}
    for group in sorted(group_labels.unique()):
        mask = group_labels == group
        rates[group] = y_pred_bin[mask].mean()
    return pd.Series(rates)

In [ ]:
# ================================================================================
# CELL 6 — Attach demographic labels to Model 3 predictions
# ================================================================================
# Model 3 already has sensitive attributes saved in its predictions CSV
# (added in Model3 Cell 12). For Models 1 and 2 we attach demographic
# data by sampling from the clinical dataset proportionally.

def attach_demographics(pred_df, clinical_df, n_samples=None):
    """
    Attach demographic columns to prediction DataFrame.
    When datasets don't share IDs, we sample demographic distributions
    from the clinical dataset to match the prediction set size.
    """
    n = n_samples or len(pred_df)
    # Sample with replacement to match size
    demo_sample = clinical_df[SENSITIVE_COLS].sample(
        n=n, replace=True, random_state=42
    ).reset_index(drop=True)

    result = pred_df.copy().reset_index(drop=True)
    for col in SENSITIVE_COLS:
        if col in demo_sample.columns:
            result[col] = demo_sample[col].values
    return result


# Attach demographics to Model 1 and 2
if m1_preds is not None:
    m1_preds = attach_demographics(m1_preds, df_clinical)
if m2_preds is not None:
    m2_preds = attach_demographics(m2_preds, df_clinical)

# Model 3 already has sensitive attributes from its training
if m3_preds is not None:
    for col in SENSITIVE_COLS:
        scol = f"sensitive_{col}"
        if scol in m3_preds.columns:
            m3_preds[col] = m3_preds[scol]

print("Demographics attached to all model predictions.")

In [ ]:
# ================================================================================
# CELL 7 — Compute fairness metrics for all models
# ================================================================================
# Compute FPR, FNR and demographic parity for each model across
# Gender, Ethnicity and Education groups.

model_data = {
    "Model 1 CNN":        m1_preds,
    "Model 2 EfficientNet": m2_preds,
    "Model 3 Hybrid":     m3_preds,
}

fairness_results = {}

for model_name, pred_df in model_data.items():
    if pred_df is None:
        continue

    print("\n" + "=" * 55)
    print(f"  {model_name} — Fairness Metrics")
    print("=" * 55)

    y_true = pred_df["true_class"].values
    y_pred = pred_df["predicted_class"].values

    model_results = {}

    for col in ["Gender", "Ethnicity", "EducationLevel"]:
        if col not in pred_df.columns:
            continue

        groups = pred_df[col]
        metrics = compute_fairness_metrics(y_true, y_pred, groups, col)
        model_results[col] = metrics

        print(f"\n  {col} breakdown:")
        print(metrics.to_string())
        print()
        check_kpi(metrics, FPR_THRESHOLD, "FPR")
        check_kpi(metrics, FNR_THRESHOLD, "FNR")

    fairness_results[model_name] = model_results


# ================================================================================
# CELL 8 — Visualise FPR and FNR across groups
# ================================================================================
# Bar charts showing FPR and FNR for each demographic group.
# We want these bars to be as equal as possible — large differences = bias.

def plot_fairness_bars(fairness_results, attr, save_name):
    """Plot FPR and FNR per group for all models side by side."""
    models_with_attr = {k: v[attr] for k, v in fairness_results.items()
                        if attr in v}
    if not models_with_attr:
        return

    n_models = len(models_with_attr)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for metric_idx, metric in enumerate(["FPR", "FNR"]):
        ax = axes[metric_idx]
        x  = None

        for i, (model_name, df) in enumerate(models_with_attr.items()):
            groups = df.index.tolist()
            values = df[metric].values
            x      = np.arange(len(groups))
            width  = 0.25
            ax.bar(x + i * width, values, width,
                   label=model_name, edgecolor="white")

        ax.axhline(FPR_THRESHOLD if metric == "FPR" else FNR_THRESHOLD,
                   color="red", linestyle="--", linewidth=1.5,
                   label=f"KPI limit ({FPR_THRESHOLD})")
        ax.set_title(f"{metric} by {attr}", fontweight="bold")
        ax.set_ylabel(metric)
        ax.set_xticks(x + width)
        ax.set_xticklabels(groups, rotation=15)
        ax.legend(fontsize=8)
        ax.set_ylim(0, max(0.3, ax.get_ylim()[1]))

    plt.suptitle(f"Fairness Metrics by {attr} — All Models",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    logger.info("Saved: %s", save_name)


plot_fairness_bars(fairness_results, "Gender",
                   "fairness_01_gender_fpr_fnr.png")
plot_fairness_bars(fairness_results, "Ethnicity",
                   "fairness_02_ethnicity_fpr_fnr.png")
plot_fairness_bars(fairness_results, "EducationLevel",
                   "fairness_03_education_fpr_fnr.png")

In [ ]:
# ================================================================================
# CELL 9 — Demographic parity analysis
# ================================================================================
# Demographic parity: the rate at which each group is predicted as positive
# (diagnosed) should be similar. The 80% rule says the lowest group rate
# should be at least 80% of the highest group rate.

print("\n" + "=" * 55)
print("  DEMOGRAPHIC PARITY ANALYSIS")
print("=" * 55)

for model_name, pred_df in model_data.items():
    if pred_df is None:
        continue

    print(f"\n  {model_name}:")
    y_pred_bin = (pred_df["predicted_class"].values > 0).astype(int)

    for col in ["Gender", "Ethnicity"]:
        if col not in pred_df.columns:
            continue
        rates = demographic_parity(y_pred_bin, pred_df[col])
        min_rate = rates.min()
        max_rate = rates.max()
        di_ratio = min_rate / max_rate if max_rate > 0 else 0

        print(f"    {col}: min={min_rate:.3f}  max={max_rate:.3f}  "
              f"DI ratio={di_ratio:.3f}  "
              f"{'✅ FAIR (≥0.8)' if di_ratio >= 0.8 else '❌ BIASED (<0.8)'}")


In [ ]:

# ================================================================================
# CELL 10 — Bias mitigation using threshold adjustment
# ================================================================================
# We apply post-processing bias mitigation by adjusting the decision threshold
# per demographic group. Instead of using 0.5 for everyone, we set a lower
# threshold for groups with high FNR (missing too many diagnoses) and a higher
# threshold for groups with high FPR (over-diagnosing).
# This is the simplest and most transparent mitigation technique.

print("\n" + "=" * 55)
print("  BIAS MITIGATION — Threshold Adjustment")
print("=" * 55)

def threshold_adjustment_mitigation(pred_df, sensitive_col, base_threshold=0.5):
    """
    Adjust prediction threshold per demographic group to equalise FNR.
    Groups with high FNR get a lower threshold (easier to predict positive).
    Groups with low FNR get a higher threshold (harder to predict positive).
    Returns updated predictions after threshold adjustment.
    """
    if sensitive_col not in pred_df.columns:
        return pred_df

    y_true     = pred_df["true_class"].values
    # Use probability of being any demented class
    prob_cols  = [c for c in pred_df.columns if c.startswith("prob_")
                  and "NonDemented" not in c]

    if not prob_cols:
        return pred_df

    prob_positive = pred_df[prob_cols].sum(axis=1).values
    groups        = pred_df[sensitive_col]
    updated_preds = pred_df["predicted_class"].copy().values

    # Compute FNR per group with base threshold
    group_fnrs = {}
    for group in sorted(groups.unique()):
        mask   = (groups == group).values
        yt_bin = (y_true[mask] > 0).astype(int)
        yp_bin = (prob_positive[mask] > base_threshold).astype(int)
        fn     = ((yt_bin == 1) & (yp_bin == 0)).sum()
        tp     = ((yt_bin == 1) & (yp_bin == 1)).sum()
        group_fnrs[group] = fn / (fn + tp) if (fn + tp) > 0 else 0

    mean_fnr = np.mean(list(group_fnrs.values()))
    print(f"\n  Adjusting thresholds for {sensitive_col}:")

    for group in sorted(groups.unique()):
        mask = (groups == group).values
        fnr  = group_fnrs[group]
        # Lower threshold for high-FNR groups, raise for low-FNR groups
        adjustment = (fnr - mean_fnr) * 0.3
        new_thresh = np.clip(base_threshold - adjustment, 0.2, 0.8)
        print(f"    {group}: FNR={fnr:.3f}  threshold {base_threshold:.2f} → {new_thresh:.2f}")

        # Apply new threshold to this group
        group_probs = prob_positive[mask]
        updated_preds[mask] = np.where(group_probs > new_thresh, 1, 0)

    return updated_preds


# Apply mitigation to each model
mitigated_results = {}

for model_name, pred_df in model_data.items():
    if pred_df is None:
        continue

    print(f"\n  {model_name}:")
    for col in ["Gender", "Ethnicity"]:
        if col not in pred_df.columns:
            continue

        mitigated_preds = threshold_adjustment_mitigation(pred_df, col)
        original_preds  = pred_df["predicted_class"].values
        y_true          = pred_df["true_class"].values

        orig_acc = accuracy_score(y_true, original_preds)
        mit_acc  = accuracy_score(y_true, mitigated_preds)

        mitigated_results[f"{model_name}_{col}"] = {
            "original_preds":  original_preds,
            "mitigated_preds": mitigated_preds,
            "y_true":          y_true,
            "groups":          pred_df[col],
        }

In [ ]:
# ================================================================================
# CELL 11 — Before vs after mitigation comparison
# ================================================================================
# Show fairness metrics before and after threshold adjustment mitigation
# to demonstrate a measurable improvement — required by the project KPIs.

print("\n" + "=" * 55)
print("  BEFORE vs AFTER MITIGATION")
print("=" * 55)

comparison_rows = []

for key, data in mitigated_results.items():
    model_col = key.split("_")
    y_true    = data["y_true"]
    groups    = data["groups"]

    for label, preds in [("Before", data["original_preds"]),
                          ("After",  data["mitigated_preds"])]:
        metrics = compute_fairness_metrics(y_true, preds, groups, "")
        fpr_gap = metrics["FPR"].max() - metrics["FPR"].min()
        fnr_gap = metrics["FNR"].max() - metrics["FNR"].min()
        acc     = accuracy_score(y_true, preds)

        comparison_rows.append({
            "Model/Group": key,
            "Stage":       label,
            "FPR Gap":     round(fpr_gap, 4),
            "FNR Gap":     round(fnr_gap, 4),
            "Accuracy":    round(acc, 4),
        })

df_comparison = pd.DataFrame(comparison_rows)
print(df_comparison.to_string(index=False))

# Plot improvement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, metric in enumerate(["FPR Gap", "FNR Gap"]):
    pivot = df_comparison.pivot(index="Model/Group",
                                columns="Stage", values=metric)
    pivot.plot(kind="bar", ax=axes[i],
               color=["#F44336", "#4CAF50"], edgecolor="white")
    axes[i].axhline(0.05, color="red", linestyle="--",
                    linewidth=1.5, label="KPI limit (0.05)")
    axes[i].set_title(f"{metric} — Before vs After Mitigation",
                      fontweight="bold")
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis="x", rotation=15)
    axes[i].legend(fontsize=8)

plt.suptitle("Bias Mitigation — Before vs After Threshold Adjustment",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fairness_04_mitigation_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ================================================================================
# CELL 12 — Equalized Odds analysis
# ================================================================================
# Equalized odds requires that both TPR and FPR are equal across groups.
# We compute equalized odds difference — lower is fairer.

print("\n" + "=" * 55)
print("  EQUALIZED ODDS ANALYSIS")
print("=" * 55)

def equalized_odds_diff(y_true, y_pred, groups):
    """
    Compute the equalized odds difference across groups.
    Lower value = more fair.
    """
    y_true_bin = (np.array(y_true) > 0).astype(int)
    y_pred_bin = (np.array(y_pred) > 0).astype(int)

    tpr_vals, fpr_vals = [], []
    for group in sorted(pd.Series(groups).unique()):
        mask = pd.Series(groups) == group
        yt   = y_true_bin[mask]
        yp   = y_pred_bin[mask]
        if len(yt) < 5:
            continue
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr_vals.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fpr_vals.append(fp / (fp + tn) if (fp + tn) > 0 else 0)

    tpr_diff = max(tpr_vals) - min(tpr_vals) if tpr_vals else 0
    fpr_diff = max(fpr_vals) - min(fpr_vals) if fpr_vals else 0
    return (tpr_diff + fpr_diff) / 2


for model_name, pred_df in model_data.items():
    if pred_df is None:
        continue
    print(f"\n  {model_name}:")
    for col in ["Gender", "Ethnicity"]:
        if col not in pred_df.columns:
            continue
        eod = equalized_odds_diff(
            pred_df["true_class"],
            pred_df["predicted_class"],
            pred_df[col]
        )
        print(f"    {col}: Equalized Odds Diff = {eod:.4f}  "
              f"{'✅ Good (≤0.1)' if eod <= 0.1 else '⚠️  Review needed'}")

In [ ]:
# ================================================================================
# CELL 13 — Final fairness summary
# ================================================================================

print("\n" + "=" * 60)
print("  FAIRNESS EVALUATION — FINAL SUMMARY")
print("=" * 60)
print(f"""
  METRICS EVALUATED
  -----------------
  FPR per group          (KPI: max gap ≤ 5%)
  FNR per group          (KPI: max gap ≤ 5%)
  Demographic parity     (80% rule: DI ratio ≥ 0.8)
  Equalized odds         (difference ≤ 0.1 = good)

  BIAS MITIGATION APPLIED
  -----------------------
  Method : Post-processing threshold adjustment per group
  Effect : Reduces FNR gap by lowering decision threshold
           for high-FNR groups

  SAVED PLOTS
  -----------
  fairness_01_gender_fpr_fnr.png
  fairness_02_ethnicity_fpr_fnr.png
  fairness_03_education_fpr_fnr.png
  fairness_04_mitigation_comparison.png

  NEXT STEP : Explainability_SHAP_LIME_GradCAM.ipynb
""")